# Sesion 7 — T5 Pandas Avanzado
## Diplomado: Machine Learning en Seguros
### Facultad de Ciencias, UNAM · 29 de abril de 2026  (18:00 - 21:00 h)

---

**Instructor:** Eric   **Sesion:** 7 de 11 del Modulo 1

---

> **Objetivo:** Dominar las operaciones avanzadas de pandas:
> `groupby`, `apply`, `merge` y manejo de datos faltantes (NaN).
> Son las operaciones que usaras en el 80% de tu trabajo actuarial.

> **Nota:** Viernes 1 de mayo NO hay sesion (Dia del Trabajo).
> La siguiente sesion es el **sabado 2 de mayo, 07:00-11:00 h**.

## Contenido

1. [groupby — resumir por grupo](#1-groupby)
2. [.agg() — multiples funciones a la vez](#2-agg)
3. [apply — aplicar cualquier funcion](#3-apply)
4. [merge — unir dos DataFrames](#4-merge)
5. [Datos faltantes: NaN, fillna, dropna](#5-nan)
6. [Ejercicio integrador de cierre](#6-ejercicio)

---
## Datos que usaremos en toda la sesion

Primero construimos la cartera de ejemplo que usaremos en todos los ejemplos:

In [1]:
import pandas as pd
import numpy as np

# Cartera de polizas de seguros
cartera = pd.DataFrame({
    'poliza':    ['P01','P02','P03','P04','P05','P06','P07','P08'],
    'titular':   ['Ana','Luis','Maria','Carlos','Rosa','Pedro','Elena','Jorge'],
    'edad':      [28,   45,    32,     62,     38,    51,    29,    44],
    'ramo':      ['GMM','Autos','GMM','Vida','Autos','GMM','Autos','Vida'],
    'vehiculo':  [None,'Sedan',None,None,'SUV',None,'Hatchback',None],
    'prima':     [2400, 5800, 1900, 9100, 4100, 3200, 2600, 7800],
    'siniest':   [0,    1,    0,    2,    0,    1,    0,    3],
    'suma_aseg': [300_000,500_000,200_000,1_000_000,400_000,250_000,300_000,800_000],
})

print(f'Cartera: {len(cartera)} polizas, {cartera["prima"].sum():,.0f} en primas')
cartera

Cartera: 8 polizas, 36,900 en primas


,poliza,titular,edad,ramo,vehiculo,prima,siniest,suma_aseg
0,P01,Ana,28,GMM,None,2400,0,300000
1,P02,Luis,45,Autos,Sedan,5800,1,500000
2,P03,Maria,32,GMM,None,1900,0,200000
3,P04,Carlos,62,Vida,None,9100,2,1000000
4,P05,Rosa,38,Autos,SUV,4100,0,400000
5,P06,Pedro,51,GMM,None,3200,1,250000
6,P07,Elena,29,Autos,Hatchback,2600,0,300000
7,P08,Jorge,44,Vida,None,7800,3,800000


---
<a id='1-groupby'></a>
## 1. groupby — Resumir por Grupo

**El problema:** tu jefe pide la prima total por ramo.
Sin groupby tendrias que filtrar cada ramo manualmente.
Con groupby lo haces en una linea.

**Logica:** Split → Apply → Combine
```
Split:   divide el DataFrame en grupos segun la clave
Apply:   aplica una funcion a cada grupo (sum, mean, count...)
Combine: une los resultados en un nuevo DataFrame
```

In [2]:
# ── El problema sin groupby ──────────────────────────────────────────────────
# Tendrias que hacer esto por cada ramo:
prima_gmm  = cartera[cartera['ramo']=='GMM']['prima'].sum()
prima_autos= cartera[cartera['ramo']=='Autos']['prima'].sum()
prima_vida = cartera[cartera['ramo']=='Vida']['prima'].sum()
print(f'GMM: ${prima_gmm:,} | Autos: ${prima_autos:,} | Vida: ${prima_vida:,}')

# ── Con groupby — una sola linea ─────────────────────────────────────────────
print()
print('Con groupby:')
print(cartera.groupby('ramo')['prima'].sum())

GMM: $7,500 | Autos: $12,500 | Vida: $16,900

Con groupby:
ramo
Autos    12500
GMM       7500
Vida     16900
Name: prima, dtype: int64


In [3]:
# ── Funciones de agregacion basicas ──────────────────────────────────────────

# Suma
print('Prima total por ramo:')
print(cartera.groupby('ramo')['prima'].sum())
print()

# Promedio
print('Prima promedio por ramo:')
print(cartera.groupby('ramo')['prima'].mean().round(2))
print()

# Conteo de polizas
print('Numero de polizas por ramo:')
print(cartera.groupby('ramo')['poliza'].count())
print()

# Suma de siniestros
print('Siniestros totales por ramo:')
print(cartera.groupby('ramo')['siniest'].sum())

Prima total por ramo:
ramo
Autos    12500
GMM       7500
Vida     16900
Name: prima, dtype: int64

Prima promedio por ramo:
ramo
Autos    4166.67
GMM      2500.00
Vida     8450.00
Name: prima, dtype: float64

Numero de polizas por ramo:
ramo
Autos    3
GMM      3
Vida     2
Name: poliza, dtype: int64

Siniestros totales por ramo:
ramo
Autos    1
GMM      1
Vida     5
Name: siniest, dtype: int64


In [4]:
# ── Multiples columnas a la vez ──────────────────────────────────────────────
resultado = cartera.groupby('ramo')[['prima','siniest']].sum()
print(resultado)
print()

# ── Agrupar por multiples columnas ───────────────────────────────────────────
# Crear grupo de edad primero
cartera['g_edad'] = pd.cut(
    cartera['edad'],
    bins=[0,30,45,60,100],
    labels=['18-30','31-45','46-60','61+']
)

print('Prima por ramo y grupo de edad:')
print(cartera.groupby(['ramo','g_edad'])['prima'].sum())

       prima  siniest
ramo                 
Autos  12500        1
GMM     7500        1
Vida   16900        5

Prima por ramo y grupo de edad:
ramo   g_edad
Autos  18-30     2600
       31-45     9900
       46-60        0
       61+          0
GMM    18-30     2400
       31-45     1900
       46-60     3200
       61+          0
Vida   18-30        0
       31-45     7800
       46-60        0
       61+       9100
Name: prima, dtype: int64


/var/folders/k1/_jflqvn90v31fwxjw3lf6y680000gn/T/ipykernel_80267/1004004524.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(cartera.groupby(['ramo','g_edad'])['prima'].sum())


In [5]:
# ── reset_index: convertir el resultado a DataFrame normal ───────────────────
# Por defecto, el resultado de groupby tiene la clave como index
# reset_index() la convierte en columna normal



resumen = (cartera
    .groupby('ramo')['prima']
    .sum()
    .reset_index()
    .rename(columns={'prima':'prima_total'})
)

print(resumen)
# Ahora podemos agregar columna de porcentaje
resumen['pct_cartera'] = (resumen['prima_total'] / resumen['prima_total'].sum() * 100).round(1)

print(resumen.to_string())

    ramo  prima_total
0  Autos        12500
1    GMM         7500
2   Vida        16900
    ramo  prima_total  pct_cartera
0  Autos        12500         33.9
1    GMM         7500         20.3
2   Vida        16900         45.8


---
<a id='2-agg'></a>
## 2. .agg() — Multiples Funciones a la Vez

`.agg()` te permite calcular varias estadisticas en una sola llamada,
con nombres de columna personalizados.

In [7]:
# ── .agg() basico ────────────────────────────────────────────────────────────
resumen = cartera.groupby('ramo').agg(
    polizas    = ('poliza',  'count'),
    prima_total= ('prima',   'sum'),
    prima_prom = ('prima',   'mean'),
    prima_max  = ('prima',   'max'),
    siniest_tot= ('siniest', 'sum'),
).round(2).reset_index()

print(resumen.to_string())

    ramo  polizas  prima_total  prima_prom  prima_max  siniest_tot
0  Autos        3        12500     4166.67       5800            1
1    GMM        3         7500     2500.00       3200            1
2   Vida        2        16900     8450.00       9100            5


In [6]:
# ── .agg() con funcion propia ────────────────────────────────────────────────
# Puedes pasar cualquier funcion como agregador

def rango(x):
    return x.max() - x.min()

def coef_variacion(x):
    return x.std() / x.mean() if x.mean() != 0 else 0

analisis = cartera.groupby('ramo')['prima'].agg([
    'count', 'sum', 'mean', 'std', rango, coef_variacion
]).round(2)

print(analisis)

       count    sum     mean      std  rango  coef_variacion
ramo                                                        
Autos      3  12500  4166.67  1601.04   3200            0.38
GMM        3   7500  2500.00   655.74   1300            0.26
Vida       2  16900  8450.00   919.24   1300            0.11


In [9]:
# ── Reporte ejecutivo completo ───────────────────────────────────────────────
reporte = cartera.groupby('ramo').agg(
    polizas    = ('poliza',  'count'),
    prima_total= ('prima',   'sum'),
    prima_prom = ('prima',   'mean'),
    siniest_tot= ('siniest', 'sum'),
).round(2).reset_index()

reporte['pct_prima']= (reporte['prima_total']/reporte['prima_total'].sum()*100).round(1)
reporte['frec_prom'] = (reporte['siniest_tot']/reporte['polizas']).round(4)

print('=' * 72)
print('  REPORTE POR RAMO — CARTERA DIPLOMADO')
print('=' * 72)
print(reporte.to_string(index=False))
print('=' * 72)
print(f'  TOTAL  {reporte["polizas"].sum():>8}  ${reporte["prima_total"].sum():>10,.2f}')

  REPORTE POR RAMO — CARTERA DIPLOMADO
 ramo  polizas  prima_total  prima_prom  siniest_tot  pct_prima  frec_prom
Autos        3        12500     4166.67            1       33.9     0.3333
  GMM        3         7500     2500.00            1       20.3     0.3333
 Vida        2        16900     8450.00            5       45.8     2.5000
  TOTAL         8  $ 36,900.00


---
<a id='3-apply'></a>
## 3. apply — Aplicar Cualquier Funcion

`.apply()` recorre cada elemento, columna o fila y aplica una funcion.
Es mas lento que las operaciones vectorizadas, pero te permite usar
**cualquier funcion** — incluyendo las de `mi_modulo.py`.

| Forma | Descripcion | Cuando usar |
|-------|-------------|-------------|
| `serie.apply(func)` | Aplica a cada elemento | Transformar una columna |
| `df.apply(func, axis=1)` | Aplica a cada fila | Logica que necesita varias columnas |

In [ ]:
# ── apply sobre una columna (Serie) ─────────────────────────────────────────
from mi_modulo import clasificar_riesgo, grupo_edad

# Con funcion de mi_modulo
cartera['riesgo'] = cartera['siniest'].apply(clasificar_riesgo) #tranforma la columna
cartera['grupo']  = cartera['edad'].apply(grupo_edad)

print(cartera[['titular','edad','grupo','siniest','riesgo']].to_string(index=False))

titular  edad grupo  siniest riesgo
    Ana    28 18-30        0   BAJO
   Luis    45 31-45        1  MEDIO
  Maria    32 31-45        0   BAJO
 Carlos    62   61+        2  MEDIO
   Rosa    38 31-45        0   BAJO
  Pedro    51 46-60        1  MEDIO
  Elena    29 18-30        0   BAJO
  Jorge    44 31-45        3   ALTO


In [12]:
# ── apply con lambda para logica rapida ──────────────────────────────────────

# Clasificar prima
cartera['nivel_prima'] = cartera['prima'].apply(
    lambda p: 'Alta' if p >= 7000 else 'Media' if p >= 3000 else 'Baja'
)

# Formatear para reporte
cartera['prima_fmt'] = cartera['prima'].apply(lambda p: f'${p:,.0f}')

print(cartera[['titular','prima','nivel_prima', 'prima_fmt']].to_string(index=False))

titular  prima nivel_prima prima_fmt
    Ana   2400        Baja    $2,400
   Luis   5800       Media    $5,800
  Maria   1900        Baja    $1,900
 Carlos   9100        Alta    $9,100
   Rosa   4100       Media    $4,100
  Pedro   3200       Media    $3,200
  Elena   2600        Baja    $2,600
  Jorge   7800        Alta    $7,800


In [ ]:
# ── apply sobre filas (axis=1) ───────────────────────────────────────────────
# La funcion recibe CADA FILA como una Serie (como un dict)

tarifas_vehiculo = {
    'Sedan':    0.025,
    'SUV':      0.035,
    'Hatchback':0.022,
}

def calcular_prima_autos(fila):
    if fila['ramo'] != 'Autos':
        return None  # solo aplica a autos
    tasa = tarifas_vehiculo.get(fila['vehiculo'], 0.030)
    return fila['suma_aseg'] * tasa * 1.16

cartera['prima_calc_autos'] = cartera.apply(calcular_prima_autos, axis=1)# axis=1 recorre fila por fila 

# Ver solo las polizas de autos
print(cartera.to_string(index=False))
autos = cartera[cartera['ramo']=='Autos'][['titular','vehiculo','suma_aseg','prima_calc_autos']]
print(autos.to_string(index=False))

poliza titular  edad  ramo  vehiculo  prima  siniest  suma_aseg g_edad  prima_calc_autos
   P01     Ana    28   GMM      None   2400        0     300000  18-30               NaN
   P02    Luis    45 Autos     Sedan   5800        1     500000  31-45           14500.0
   P03   Maria    32   GMM      None   1900        0     200000  31-45               NaN
   P04  Carlos    62  Vida      None   9100        2    1000000    61+               NaN
   P05    Rosa    38 Autos       SUV   4100        0     400000  31-45           16240.0
   P06   Pedro    51   GMM      None   3200        1     250000  46-60               NaN
   P07   Elena    29 Autos Hatchback   2600        0     300000  18-30            7656.0
   P08   Jorge    44  Vida      None   7800        3     800000  31-45               NaN
titular  vehiculo  suma_aseg  prima_calc_autos
   Luis     Sedan     500000           14500.0
   Rosa       SUV     400000           16240.0
  Elena Hatchback     300000            7656.0


In [14]:
# ── Cuando usar apply vs operacion vectorizada ───────────────────────────────

# VECTORIZADA (rapida, preferida cuando es posible):
cartera['prima_con_iva'] = cartera['prima'] * 1.16
cartera['prima_mensual'] = cartera['prima'] / 12

# APPLY (cuando necesitas logica compleja):
# cartera['riesgo'] = cartera['siniest'].apply(clasificar_riesgo)

# Regla practica:
# Si puedes escribirlo como df['col'] * valor → vectorizada
# Si necesitas if/else o varias columnas → apply

print('Prima con IVA (vectorizada):')
print(cartera[['titular','prima','prima_con_iva']].head(3).to_string(index=False))

Prima con IVA (vectorizada):
titular  prima  prima_con_iva
    Ana   2400         2784.0
   Luis   5800         6728.0
  Maria   1900         2204.0


---
<a id='4-merge'></a>
## 4. merge — Unir Dos DataFrames

En seguros los datos viven en tablas separadas: cartera, tarifas,
siniestros, catalogos de agentes... `merge()` las une por una clave comun.

Es el equivalente del **BUSCARV** de Excel — pero para tablas completas.

In [8]:
# ── Preparar los DataFrames ──────────────────────────────────────────────────
# Cartera simplificada
polizas = pd.DataFrame({
    'poliza':  ['P01','P02','P03','P04','P05'],
    'titular': ['Ana','Luis','Maria','Carlos','Rosa'],
    'ramo_id': ['R01','R02','R01','R03','R02'],
    'prima':   [2400, 5800, 1900, 9100, 4100],
})

# Catalogo de ramos
ramos = pd.DataFrame({
    'ramo_id':     ['R01',   'R02',    'R03'],
    'nombre_ramo': ['GMM',   'Autos',  'Vida'],
    'cobertura':   [500_000, 300_000, 1_000_000],
})

print('Polizas:')
print(polizas)
print()
print('Ramos:')
print(ramos)

Polizas:
  poliza titular ramo_id  prima
0    P01     Ana     R01   2400
1    P02    Luis     R02   5800
2    P03   Maria     R01   1900
3    P04  Carlos     R03   9100
4    P05    Rosa     R02   4100

Ramos:
  ramo_id nombre_ramo  cobertura
0     R01         GMM     500000
1     R02       Autos     300000
2     R03        Vida    1000000


In [16]:
# ── left merge — el mas usado en actuaria ────────────────────────────────────
# Todas las polizas + la info del ramo donde coincida

resultado = pd.merge(polizas, ramos, on='ramo_id', how='left')
print('Resultado left merge:')
print(resultado.to_string(index=False))
print(f'Filas: polizas={len(polizas)}, resultado={len(resultado)} (se conservan todas)')

Resultado left merge:
poliza titular ramo_id  prima nombre_ramo  cobertura
   P01     Ana     R01   2400         GMM     500000
   P02    Luis     R02   5800       Autos     300000
   P03   Maria     R01   1900         GMM     500000
   P04  Carlos     R03   9100        Vida    1000000
   P05    Rosa     R02   4100       Autos     300000
Filas: polizas=5, resultado=5 (se conservan todas)


In [17]:
# ── Diferencia entre left, inner y outer ─────────────────────────────────────
# Agregar una poliza sin ramo_id valido para ver la diferencia
polizas_extra = pd.concat([polizas,
    pd.DataFrame([{'poliza':'P06','titular':'Pedro','ramo_id':'R99','prima':3200}])
], ignore_index=True)

# LEFT: conserva todas las polizas, NaN donde no hay ramo
left = pd.merge(polizas_extra, ramos, on='ramo_id', how='left')
print(f'left:  {len(left)} filas  (conserva P06 con NaN en nombre_ramo)')
print(left[['poliza','ramo_id','nombre_ramo']].to_string(index=False))
print()

# INNER: solo polizas con ramo valido
inner = pd.merge(polizas_extra, ramos, on='ramo_id', how='inner')
print(f'inner: {len(inner)} filas  (descarta P06 porque R99 no existe)')

left:  6 filas  (conserva P06 con NaN en nombre_ramo)
poliza ramo_id nombre_ramo
   P01     R01         GMM
   P02     R02       Autos
   P03     R01         GMM
   P04     R03        Vida
   P05     R02       Autos
   P06     R99         NaN

inner: 5 filas  (descarta P06 porque R99 no existe)


In [9]:
# ── Claves con nombres distintos en cada tabla ───────────────────────────────
# A veces las tablas usan nombres distintos para la misma clave

polizas2 = pd.DataFrame({
    'id_poliza': ['P01','P02','P03'],
    'titular':   ['Ana','Luis','Maria'],
    'codigo_ramo':['R01','R02','R01'],
})

ramos2 = pd.DataFrame({
    'id_ramo':  ['R01','R02','R03'],
    'ramo_nombre':['GMM','Autos','Vida'],
})

# left_on = columna en el DataFrame izquierdo
# right_on = columna en el DataFrame derecho
res = pd.merge(polizas2, ramos2,
               left_on='codigo_ramo',
               right_on='id_ramo',
               how='left')
print(res[['id_poliza','titular','codigo_ramo','ramo_nombre']].to_string(index=False))

id_poliza titular codigo_ramo ramo_nombre
      P01     Ana         R01         GMM
      P02    Luis         R02       Autos
      P03   Maria         R01         GMM


---
<a id='5-nan'></a>
## 5. Datos Faltantes — NaN

**NaN** (Not a Number) representa un valor faltante.
Aparece cuando: lees un CSV con celdas vacias,
haces un merge sin coincidencia, o calculas algo imposible.

**Regla antes de modelar:** siempre revisa y trata los NaN.
Un modelo de ML con NaN en los datos de entrada falla o da resultados incorrectos.

In [11]:
# ── Crear datos con NaN para practicar ───────────────────────────────────────
df = pd.DataFrame({
    'poliza':   ['P01','P02','P03','P04','P05','P06'],
    'ramo':     ['GMM', None, 'Autos', 'GMM', None, 'Vida'],
    'prima':    [2400, np.nan, 5800, np.nan, 4100, 9100],
    'siniest':  [0, 1, np.nan, 2, 0, np.nan],
    'agente':   ['Ag1','Ag2','Ag1',None,'Ag3','Ag2'],
})

print('DataFrame con valores faltantes:')
print(df)
print()

# Cuantos NaN por columna
print('NaN por columna:')
print(df.isna().sum())
print()

# Porcentaje de NaN
print('Porcentaje de NaN:')
print((df.isna().mean()*100).round(1))

DataFrame con valores faltantes:
  poliza   ramo   prima  siniest agente
0    P01    GMM  2400.0      0.0    Ag1
1    P02   None     NaN      1.0    Ag2
2    P03  Autos  5800.0      NaN    Ag1
3    P04    GMM     NaN      2.0   None
4    P05   None  4100.0      0.0    Ag3
5    P06   Vida  9100.0      NaN    Ag2

NaN por columna:
poliza     0
ramo       2
prima      2
siniest    2
agente     1
dtype: int64

Porcentaje de NaN:
poliza      0.0
ramo       33.3
prima      33.3
siniest    33.3
agente     16.7
dtype: float64


In [12]:
# ── fillna: rellenar valores faltantes ───────────────────────────────────────

# Rellenar con valor fijo
print('Rellenar siniest con 0:')
print(df['siniest'].fillna(0))
print()

# Rellenar con la mediana (para numericos con outliers)
mediana_prima = df['prima'].median()
print(f'Mediana prima: ${mediana_prima:,.2f}')
df_limpio = df.copy()
df_limpio['prima'] = df_limpio['prima'].fillna(mediana_prima)
df_limpio['siniest'] = df_limpio['siniest'].fillna(0)
df_limpio['ramo'] = df_limpio['ramo'].fillna('Sin ramo')
df_limpio['agente'] = df_limpio['agente'].fillna('Sin asignar')

print('Despues de fillna:')
print(df_limpio.isna().sum())

Rellenar siniest con 0:
0    0.0
1    1.0
2    0.0
3    2.0
4    0.0
5    0.0
Name: siniest, dtype: float64

Mediana prima: $4,950.00
Despues de fillna:
poliza     0
ramo       0
prima      0
siniest    0
agente     0
dtype: int64


In [21]:
# ── dropna: eliminar filas con NaN ───────────────────────────────────────────

print(f'Total filas originales: {len(df)}')

# Eliminar si CUALQUIER columna tiene NaN
sin_nan = df.dropna()
print(f'Despues de dropna(): {len(sin_nan)} filas')

# Eliminar solo si NaN en columnas criticas
solo_prima = df.dropna(subset=['prima'])
print(f'Dropna solo prima: {len(solo_prima)} filas')

# Ver que filas tienen NaN
print()
print('Filas con algun NaN:')
print(df[df.isna().any(axis=1)])

Total filas originales: 6
Despues de dropna(): 1 filas
Dropna solo prima: 4 filas

Filas con algun NaN:
  poliza   ramo   prima  siniest agente
1    P02   None     NaN      1.0    Ag2
2    P03  Autos  5800.0      NaN    Ag1
3    P04    GMM     NaN      2.0   None
4    P05   None  4100.0      0.0    Ag3
5    P06   Vida  9100.0      NaN    Ag2


In [13]:
# ── Duplicados ───────────────────────────────────────────────────────────────
df_con_dup = pd.concat([df, df.iloc[:2]], ignore_index=True)

print(f'Con duplicados: {len(df_con_dup)} filas')
print(f'Duplicados detectados: {df_con_dup.duplicated().sum()}')

# Eliminar duplicados
df_sin_dup = df_con_dup.drop_duplicates()
print(f'Sin duplicados: {len(df_sin_dup)} filas')

# Duplicados por columna especifica
df_sin_dup2 = df_con_dup.drop_duplicates(subset=['poliza'])
print(f'Sin duplicados por poliza: {len(df_sin_dup2)} filas')

Con duplicados: 8 filas
Duplicados detectados: 2
Sin duplicados: 6 filas
Sin duplicados por poliza: 6 filas


---
<a id='6-ejercicio'></a>
## 6. Ejercicio Integrador de Cierre

Combina todo lo visto en la sesion: `groupby`, `apply`, `merge` y manejo de NaN.

In [14]:
# ── Datos con imperfecciones reales ─────────────────────────────────────────
import pandas as pd
import numpy as np
from mi_modulo import clasificar_riesgo

cartera_ej = pd.DataFrame({
    'poliza':  ['A01','A02','A03','A04','A05','A06'],
    'nombre':  ['Sofia','Miguel','Laura','Roberto','Carmen','Pedro'],
    'ramo_id': ['R01','R02','R01','R03','R02','R01'],
    'prima':   [3200, 5800, np.nan, 9100, 4100, 2600],
    'siniest': [0, 1, 0, 2, 0, np.nan],
    'suma':    [300_000,500_000,250_000,600_000,400_000,280_000],
})

catalogo_ramos = pd.DataFrame({
    'ramo_id':    ['R01',   'R02',   'R03'],
    'nombre_ramo':['GMM',   'Autos', 'Vida'],
    'tasa':       [0.022,   0.035,   0.018],
})
print(f'Datos cargados: {len(cartera_ej)} polizas')

Datos cargados: 6 polizas


In [17]:
# ── Tarea 1: Detectar y tratar NaN ───────────────────────────────────────────
# Muestra cuantos NaN hay por columna
# Rellena prima con la mediana y siniest con 0

# Tu codigo aqui:
print("Numero de NaN por columna")
print(cartera_ej.isna().sum())

mediana = cartera_ej['prima'].median()
cartera_limpia = cartera_ej.copy()
cartera_limpia['prima'] = df_limpio['prima'].fillna(mediana)
cartera_limpia['siniest'] = df_limpio['siniest'].fillna(0)

Numero de NaN por columna
poliza     0
nombre     0
ramo_id    0
prima      1
siniest    1
suma       0
dtype: int64


In [18]:
# ── Tarea 2: Merge con catalogo de ramos ─────────────────────────────────────
# Haz un left merge de cartera_ej con catalogo_ramos usando 'ramo_id'
# Verifica que el resultado tiene las columnas nombre_ramo y tasa

# Tu codigo aqui:
cartera2 = pd.merge(cartera_limpia,
    catalogo_ramos, on='ramo_id', how='left'
)
cartera2

,poliza,nombre,ramo_id,prima,siniest,suma,nombre_ramo,tasa
0,A01,Sofia,R01,2400.0,0.0,300000,GMM,0.022
1,A02,Miguel,R02,4950.0,1.0,500000,Autos,0.035
2,A03,Laura,R01,5800.0,0.0,250000,GMM,0.022
3,A04,Roberto,R03,4950.0,2.0,600000,Vida,0.018
4,A05,Carmen,R02,4100.0,0.0,400000,Autos,0.035
5,A06,Pedro,R01,9100.0,0.0,280000,GMM,0.022


In [21]:
# ── Tarea 3: Columnas derivadas con apply ────────────────────────────────────
# Crea la columna 'riesgo' usando clasificar_riesgo() de mi_modulo
# Crea la columna 'prima_calc' = suma * tasa * 1.16 (usando apply axis=1)
import mi_modulo
# Tu codigo aqui:
cartera2['riesgo'] = cartera2['siniest'].apply(mi_modulo.clasificar_riesgo)

cartera2['prima_calc'] = cartera2.apply(lambda f: f['suma']*f['tasa']*1.16, axis=1 )
cartera2

,poliza,nombre,ramo_id,prima,siniest,suma,nombre_ramo,tasa,riesgo,prima_calc
0,A01,Sofia,R01,2400.0,0.0,300000,GMM,0.022,BAJO,7656.0
1,A02,Miguel,R02,4950.0,1.0,500000,Autos,0.035,MEDIO,20300.0
2,A03,Laura,R01,5800.0,0.0,250000,GMM,0.022,BAJO,6380.0
3,A04,Roberto,R03,4950.0,2.0,600000,Vida,0.018,MEDIO,12528.0
4,A05,Carmen,R02,4100.0,0.0,400000,Autos,0.035,BAJO,16240.0
5,A06,Pedro,R01,9100.0,0.0,280000,GMM,0.022,BAJO,7145.6


In [ ]:
# ── Tarea 4: Reporte con groupby ────────────────────────────────────────────
# Con groupby + .agg() calcula por nombre_ramo:
# - polizas: count
# - prima_total: sum
# - prima_prom: mean
# - siniest_total: sum
# Agrega columna pct_cartera = prima del ramo / total * 100

# Tu codigo aqui:

agrup1 = cartera2.groupby('nombre_ramo').agg(
    polizas = ('poliza', 'count'),
    prima_total = ('prima', 'sum'),
    prima_prom = ('prima', 'mean'),
    siniest_total = ('siniest', 'sum')
).round(1).reset_index()
agrup1['pct_cartera'] = (agrup1['prima_total']/agrup1['prima_total'].sum() * 100).round(1)
agrup1

,nombre_ramo,polizas,prima_total,prima_prom,siniest_total,pct_cartera
0,Autos,2,9050.0,4525.0,1.0,28.9
1,GMM,3,17300.0,5766.7,0.0,55.3
2,Vida,1,4950.0,4950.0,2.0,15.8


---
## Resumen de la Sesion 7

| Concepto | Lo que aprendimos |
|---------|------------------|
| **groupby** | Split-Apply-Combine en una linea |
| **.agg()** | Multiples funciones: count, sum, mean, std y propias |
| **apply (Serie)** | Aplicar funcion a cada elemento de una columna |
| **apply (axis=1)** | Aplicar funcion a cada fila — usa varias columnas |
| **merge** | left/inner/outer — unir por clave comun |
| **NaN** | .isna().sum(), .fillna(), .dropna(subset=) |
| **Duplicados** | .duplicated(), .drop_duplicates(subset=) |

**Proxima sesion — Sab 2 de mayo, 07:00-11:00 h (4 hrs):**
T5 continua: leer CSV/Excel/Parquet, `pivot_table`, optimizacion con `dtypes`.

**Tarea:**
```bash
git add sesion7_M1_notebook.ipynb
git commit -m "Sesion 7: groupby apply merge NaN"
git push
```

---
*Diplomado ML en Seguros · Facultad de Ciencias, UNAM · 2026*